# W7: 재고 알림 자동화

재고 임계치 미만 상품 감지 후 웹훅 전송 실패 시 콘솔로 fallback 처리합니다.


In [ ]:
import io
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

try:
    import google.generativeai as genai
except Exception:
    genai = None

GEMINI_MODEL = "gemini-2.0-flash"

for font_path in fm.findSystemFonts():
    try:
        fm.fontManager.addfont(font_path)
    except Exception:
        continue


def safe_upload():
    try:
        from google.colab import files
    except Exception:
        print("Colab 환경이 아니라 files.upload를 사용할 수 없습니다.")
        return {}

    try:
        return files.upload()
    except Exception as e:
        print(f"files.upload() 취소/실패: {e}")
        print("files.upload() 취소되어 기본 데이터 사용")
        return {}


def use_gemini(prompt: str, fallback: str) -> str:
    if not genai:
        return fallback
    key = os.getenv("GOOGLE_API_KEY", "").strip()
    if not key:
        return fallback
    try:
        genai.configure(api_key=key)
        model = genai.GenerativeModel(GEMINI_MODEL)
        out = model.generate_content(prompt)
        return getattr(out, "text", "").strip() or fallback
    except Exception as e:
        print(f"Gemini 호출 실패: {e}")
        return fallback

import requests

inv = pd.DataFrame({
    "상품명": ["상품A", "상품B", "상품C", "상품D"],
    "현재재고": [12, 4, 20, 3],
    "안전재고": [10, 8, 15, 5],
    "공급업체": ["A사", "B사", "A사", "C사"]
})

def webhook_send(msg, webhook):
    if not webhook:
        print("Webhook 미설정. 콘솔 출력:")
        print(msg)
        return
    try:
        requests.post(webhook, json={"text": msg}, timeout=5)
        print("webhook 전송 완료")
    except Exception as e:
        print(f"webhook 실패: {e}")
        print(msg)

inv["상태"] = np.where(inv["현재재고"] <= inv["안전재고"], "위험", "안전")
issue = inv[inv["상태"] == "위험"]
webhook = os.getenv("SLACK_WEBHOOK_URL", "")

for _, r in issue.iterrows():
    prompt = f"상품:{r.상품명}, 현재재고:{r.현재재고}, 안전재고:{r.안전재고}, 공급업체:{r.공급업체}. 재고 알림문구 생성"
    msg = use_gemini(prompt, f"{r.상품명} 재고가 부족합니다({r.현재재고}). 긴급 보충이 필요합니다.")
    webhook_send(msg, webhook)

print(issue)
